In [ ]:
# =================================================================
# 05a_chronicle_merge
# Goal: Combine flood (positive) and no-flood (negative) samples
#       into a single unified dataset and save two output formats:
#
#   FULL        — includes 3D IMERG matrices  (for CNN-LSTM models)
#   LIGHTWEIGHT — tabular only, no matrices   (for RF / XGBoost)
#
# Run this notebook once whenever new no-flood batches are added.
# Do NOT re-run for EDA purposes — use 05b_chronicle_eda.ipynb instead.
# =================================================================

import os
import glob
import numpy as np
import pandas as pd
from shapely import wkt as shapely_wkt

# ── 1. CONFIGURATION ──────────────────────────────────────────────
NEGATIVE_BATCHES_DIR    = r"D:\Development\RESEARCH\urban_flood_database\chronicle\no_flood_event_outputs"
POSITIVE_EVENTS_PATH    = r"\\vscifs\hydrolab1\hydrolab\home\Raz\Research\chronicle\chronicle_rain_master_V07.pkl"
FULL_OUTPUT_PATH        = r"D:\Development\RESEARCH\urban_flood_database\chronicle\unified_ml_dataset_FULL.pkl"
LIGHTWEIGHT_OUTPUT_PATH = r"D:\Development\RESEARCH\urban_flood_database\chronicle\unified_ml_dataset_LIGHTWEIGHT.pkl"

# Columns excluded from the LIGHTWEIGHT version (too heavy for tabular ML)
HEAVY_COLS = ['imerg_matrix', 'imerg_mask', 'imerg_meta', 'geometry_wkt']

# Logical column order for the final dataset
LOGICAL_ORDER = [
    # Identifiers
    'uuid', 'event_id',
    # Time
    'start_time', 'end_time', 'duration_days',
    # Basin geometry and land cover
    'area_km2', 'urban_built_up_area_m2', 'urban_percentage',
    # Centroid coordinates (extracted from geometry_wkt — kept in LIGHTWEIGHT for mapping)
    'centroid_lon', 'centroid_lat',
    # Topographic and hydrological indices
    'upa_max', 'upa_p95', 'upa_p99',
    'PFDI_p99',
    # Rainfall intensities (sorted by duration)
    '30_max_rainfall_intens', '60_max_rainfall_intens',
    '120_max_rainfall_intens', '240_max_rainfall_intens',
    '360_max_rainfall_intens', '720_max_rainfall_intens',
    '1440_max_rainfall_intens',
    # Heavy data (CNN inputs — dropped in LIGHTWEIGHT)
    'imerg_type', 'imerg_meta', 'imerg_mask', 'imerg_matrix', 'geometry_wkt',
    # Target label — always last
    'is_flood',
]

# ── 2. LOAD NEGATIVE SAMPLES (CLASS 0) ───────────────────────────
print("Loading no-flood batches...")
batch_files = glob.glob(os.path.join(NEGATIVE_BATCHES_DIR, "no_flood_batch_*.pkl"))

if not batch_files:
    raise FileNotFoundError(f"No batch files found in: {NEGATIVE_BATCHES_DIR}")

negative_df = pd.concat(
    [pd.read_pickle(f) for f in batch_files], ignore_index=True
)
negative_df['is_flood'] = 0

# Make event IDs traceable: append the storm date to the original polygon ID
negative_df['start_time'] = pd.to_datetime(negative_df['start_time'])
negative_df['event_id'] = (
    negative_df['event_id'].astype(str)
    + '_noflood_'
    + negative_df['start_time'].dt.strftime('%Y%m%d')
)
print(f"  No-flood samples: {len(negative_df):,}")

# ── 3. LOAD POSITIVE SAMPLES (CLASS 1) ───────────────────────────
print("Loading flood events...")
positive_df = pd.read_pickle(POSITIVE_EVENTS_PATH)
positive_df['is_flood'] = 1
print(f"  Flood samples:    {len(positive_df):,}")

# ── 4. MERGE ─────────────────────────────────────────────────────
print("Merging...")
common_cols = list(set(positive_df.columns) & set(negative_df.columns))
unified_df  = pd.concat(
    [positive_df[common_cols], negative_df[common_cols]], ignore_index=True
)
del positive_df, negative_df

# ── 5. EXTRACT CENTROIDS ──────────────────────────────────────────
# Extract lon/lat centroid from geometry_wkt before it is dropped in LIGHTWEIGHT.
# These lightweight scalar coordinates allow spatial analysis without carrying
# the full polygon geometry.
print("Extracting centroids from geometry_wkt...")

def get_centroid(geom_wkt):
    try:
        geom = shapely_wkt.loads(geom_wkt)
        return geom.centroid.x, geom.centroid.y
    except Exception:
        return np.nan, np.nan

centroids = unified_df['geometry_wkt'].apply(get_centroid)
unified_df['centroid_lon'] = centroids.apply(lambda c: c[0])
unified_df['centroid_lat'] = centroids.apply(lambda c: c[1])
print("  Centroids extracted.")

# ── 6. REORDER COLUMNS ────────────────────────────────────────────
ordered_cols  = [c for c in LOGICAL_ORDER if c in unified_df.columns]
leftover_cols = [c for c in unified_df.columns if c not in ordered_cols]
unified_df    = unified_df[ordered_cols + leftover_cols]

n_flood    = (unified_df['is_flood'] == 1).sum()
n_no_flood = (unified_df['is_flood'] == 0).sum()
print(f"\nUnified dataset: {len(unified_df):,} rows × {len(unified_df.columns)} columns")
print(f"  Flood (1):    {n_flood:,}  ({n_flood/len(unified_df)*100:.1f}%)")
print(f"  No-flood (0): {n_no_flood:,}  ({n_no_flood/len(unified_df)*100:.1f}%)")

# ── 7. SAVE FULL DATASET (CNN / deep learning) ────────────────────
print(f"\nSaving FULL dataset → {FULL_OUTPUT_PATH}")
unified_df.to_pickle(FULL_OUTPUT_PATH)
print("  FULL saved.")

# ── 8. SAVE LIGHTWEIGHT DATASET (tabular ML) ──────────────────────
print(f"Saving LIGHTWEIGHT dataset → {LIGHTWEIGHT_OUTPUT_PATH}")
drop_cols      = [c for c in HEAVY_COLS if c in unified_df.columns]
lightweight_df = unified_df.drop(columns=drop_cols)
lightweight_df.to_pickle(LIGHTWEIGHT_OUTPUT_PATH)
print("  LIGHTWEIGHT saved.")
print(f"  Columns: {list(lightweight_df.columns)}")